In [1]:
import numpy as np
import xgboost as xgb
import optuna
import pandas as pd

# Загружаем данные
X_train = np.loadtxt('data_pizdata/X_train.csv', delimiter=',')
y_train = np.loadtxt('data_pizdata/y_train.csv', delimiter=',')

# Удаляем последние 4 столбца
X_train = np.delete(X_train, [4, 5, 6, 7], axis=1)

# Преобразуем в DMatrix для XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)

/opt/anaconda3/envs/ML/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
X_train

array([[0.4510095 , 0.61460638, 0.16730614, ..., 2.        , 6.69126301,
        6.79214501],
       [0.8371234 , 0.55844869, 0.03419815, ..., 2.        , 0.73517836,
        0.32481099],
       [0.60563576, 0.35095504, 0.67840821, ..., 0.        , 3.92152425,
        6.61085878],
       ...,
       [0.4811432 , 0.70772098, 0.88116348, ..., 2.        , 3.53998395,
        0.94240185],
       [0.69956601, 0.90805913, 0.57516987, ..., 2.        , 1.23707547,
        0.81672114],
       [0.60549189, 0.11642799, 0.08077757, ..., 1.        , 1.56143998,
        1.09768652]])

In [5]:
def objective(trial):
    # Подбираем гиперпараметры
    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'eta': trial.suggest_loguniform('eta', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'seed': 42
    }

    # Кросс-валидация XGBoost с 5 фолдами
    cv_results = xgb.cv(
        params,
        dtrain,
        num_boost_round=1000,
        nfold=5,
        early_stopping_rounds=50,
        verbose_eval=False,
        seed=42
    )

    # Возвращаем минимальное значение RMSE на валидации
    best_rmse = cv_results['test-rmse-mean'].min()
    return best_rmse

In [6]:
# Создаём исследование и подбираем параметры
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)  # число итераций можно увеличить

print("Лучшие гиперпараметры:", study.best_params)
print("Лучший RMSE:", study.best_value)

# Обучаем финальную модель с найденными гиперпараметрами
best_params = study.best_params
best_params.update({
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'seed': 42
})

# Повторная кросс-валидация для определения числа итераций
cv_results = xgb.cv(
    best_params,
    dtrain,
    num_boost_round=1000,
    nfold=5,
    early_stopping_rounds=50,
    verbose_eval=False,
    seed=42
)
best_num_boost_round = len(cv_results)
print("Оптимальное число итераций:", best_num_boost_round)

# Обучаем модель на полном наборе данных
final_model = xgb.train(best_params, dtrain, num_boost_round=best_num_boost_round)


[I 2025-04-05 12:24:38,750] A new study created in memory with name: no-name-c1348d7b-411c-476e-8fb2-02c252b14754
/var/folders/mv/lrl6yw716j18jqsjtrtfkklr0000gn/T/ipykernel_48015/2702449213.py:6: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'eta': trial.suggest_loguniform('eta', 0.01, 0.3),
/var/folders/mv/lrl6yw716j18jqsjtrtfkklr0000gn/T/ipykernel_48015/2702449213.py:8: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
/var/folders/mv/lrl6yw716j18jqsjtrtfkklr0000gn/T/ipykernel_48015/2702449213.py:9: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/op

Лучшие гиперпараметры: {'eta': 0.0266302061025319, 'max_depth': 6, 'subsample': 0.5443718414276817, 'colsample_bytree': 0.7962279212474508, 'min_child_weight': 2}
Лучший RMSE: 0.5367544841957527
Оптимальное число итераций: 1000


In [10]:
# Предсказание на тестовой выборке
X_test = np.loadtxt('data_pizdata/X_test.csv', delimiter=',')
# Удаляем последние 4 столбца из тестовых данных, чтобы их структура совпадала с тренировочными
X_test = np.delete(X_test, [4, 5, 6, 7], axis=1)
dtest = xgb.DMatrix(X_test)
predictions = final_model.predict(dtest)

# Формируем файл submission.csv
submission = pd.DataFrame({'y': predictions, 'ID': list(range(len(predictions))),})
submission.to_csv('submission_zuiki5.csv', index=False)